# Training LoRA — Dungeons & Byte

Allena un adapter LoRA su Qwen3 per UNA tipologia alla volta (bestiario_png, incantesimo, specie, classe, background, luogo, oggetto, regola).

**Prima di iniziare**: `Runtime > Change runtime type > T4 GPU`, poi carica l'intera cartella `colab/` (con `dataset/`, `scripts/`, ecc.) dentro `MyDrive/colab` sul tuo Google Drive.

Il training salva un checkpoint ogni 25 step direttamente su Drive: se la sessione Colab si disconnette, basta rieseguire questo notebook con la stessa tipologia — riprende da solo dall'ultimo checkpoint invece di ripartire da zero.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/colab'
import os
assert os.path.isdir(DRIVE_ROOT), f"Cartella non trovata: {DRIVE_ROOT} — carica prima la cartella colab/ su Drive"

In [ ]:
!pip install -q unsloth unsloth_zoo trl datasets accelerate bitsandbytes

In [ ]:
# @title Parametri training { run: "auto" }
TIPOLOGIA = "incantesimo"  # @param ["bestiario_png", "incantesimo", "specie", "classe", "background", "luogo", "oggetto", "regola"]
MODEL_NAME = "unsloth/Qwen3-4B-unsloth-bnb-4bit"  # @param ["unsloth/Qwen3-4B-unsloth-bnb-4bit", "unsloth/Qwen3-8B-unsloth-bnb-4bit"]
EPOCHS = 3  # @param {type:"integer"}
BATCH_SIZE = 2  # @param {type:"integer"}
GRAD_ACCUM = 4  # @param {type:"integer"}

In [ ]:
import sys
sys.path.append(f"{DRIVE_ROOT}/scripts")
from train_lora import run_training

result = run_training(
    tipologia=TIPOLOGIA,
    model_name=MODEL_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    drive_root=DRIVE_ROOT,
)

## Test rapido di generazione

Verifica veloce che l'adapter appena allenato produca output sensato prima di passare alla tipologia successiva.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/colab'
TIPOLOGIA = 'incantesimo'  # @param ["bestiario_png", "incantesimo", "specie", "classe", "background", "luogo", "oggetto", "regola"]

from unsloth import FastLanguageModel

adapter_dir = f"{DRIVE_ROOT}/adapters/{TIPOLOGIA}"
model, tokenizer = FastLanguageModel.from_pretrained(model_name=adapter_dir, max_seq_length=4096, load_in_4bit=True)
FastLanguageModel.for_inference(model)

# Sostituire con un testo reale non visto in training, stesso system prompt usato nel dataset
test_messages = [
    {"role": "system", "content": "Sei un motore di estrazione dati per manuali D&D. Analizza il testo fornito ed estrai tutte le entita' presenti, restituendo un array JSON."},
    {"role": "user", "content": "<incolla qui un pezzo di testo reale da testare>"},
]
inputs = tokenizer.apply_chat_template(test_messages, tokenize=True, add_generation_prompt=True, enable_thinking=False, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=500, temperature=0.1)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## (Opzionale) Export in GGUF per l'inferenza su LXC 121

Da eseguire solo sull'adapter finale scelto per la produzione — genera il file .gguf da scaricare e portare sul container `paddleocr-vl` per servirlo con `llama-server`, esattamente come il modello attuale.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/colab'
TIPOLOGIA = 'incantesimo'  # @param ["bestiario_png", "incantesimo", "specie", "classe", "background", "luogo", "oggetto", "regola"]

from unsloth import FastLanguageModel

adapter_dir = f"{DRIVE_ROOT}/adapters/{TIPOLOGIA}"
model, tokenizer = FastLanguageModel.from_pretrained(model_name=adapter_dir, max_seq_length=4096, load_in_4bit=True)

model.save_pretrained_gguf(f"{DRIVE_ROOT}/adapters/{TIPOLOGIA}_gguf", tokenizer, quantization_method="q8_0")